# XGBoost By-Gene mRNA Data Investigation

This notebook uses the strongest realistic XGBoost setup from the saved baseline notebook: by-gene evaluation, mRNA included, and frozen Optuna-tuned hyperparameters.

In [44]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Build Current Pipeline Data

This investigation uses the current shared preprocessing setup: `strict_cleaning=True`, `add_mrna=True`, and `use_normalized_conditions=False`.

Important note: the frozen XGBoost hyperparameters were tuned in the earlier baseline notebook, not re-tuned on this stricter pipeline. That is acceptable for data investigation, but the results should be treated as investigation-first rather than final model selection.

In [45]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)
X, groups, y = pipeline.prepare_for_classical_ml(
    enriched_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)

mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = enriched_df.loc[mask].reset_index(drop=True).copy()

analysis_df["patent_group"] = analysis_df.get(
    "patent_ID", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
analysis_df["Authorization_status"] = analysis_df.get(
    "Authorization_status", pd.Series(index=analysis_df.index, dtype=object)
).fillna("UNKNOWN")
analysis_df["Title"] = analysis_df.get(
    "Title", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")

print("Analysis dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## Prediction Preview

## Frozen Hyperparameters

These are the saved Optuna hyperparameters from the earlier by-gene + mRNA XGBoost notebook.

In [46]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## By-Gene Out-of-Fold Evaluation

This is the harder and more biologically realistic view. Each prediction comes from a fold where that gene group was held out, so weak slices here are stronger candidates for transfer-poisoning before deep learning.

In [47]:
frozen_params = {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.15881823130907038, 'subsample': 0.8812898741586134, 'colsample_bytree': 0.7824379872752019, 'min_child_weight': 4, 'reg_lambda': 0.8342807691178866, 'reg_alpha': 1.4296995092035882, 'gamma': 0.07531958697602548}
gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=0, random_state=42)

fold_rows = []
oof_frames = []
oof_true = []
oof_pred = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = make_model(frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    oof_true.append(y[test_idx])
    oof_pred.append(fold_pred)

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["residual"] = fold_frame["y_true"] - fold_frame["y_pred"]
    fold_frame["abs_error"] = fold_frame["residual"].abs()
    oof_frames.append(fold_frame)

    fold_rows.append({
        "fold_id": fold_id,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "n_train_groups": int(len(np.unique(groups[train_idx]))),
        "n_test_groups": int(len(np.unique(groups[test_idx]))),
    })

predictions_df = pd.concat(oof_frames, ignore_index=True)
metrics_df = pd.DataFrame([evaluate(np.concatenate(oof_true), np.concatenate(oof_pred))])
fold_summary = pd.DataFrame(fold_rows)
fold_summary


,fold_id,n_train,n_test,n_train_groups,n_test_groups
0,1,23629,11815,36,18
1,2,23631,11813,36,18
2,3,23628,11816,36,18


## Fold Summary

## Overall Metrics

In [48]:
metrics_df

,spearman,pearson,rmse,mae
0,0.376908,0.375735,33.685283,26.584346


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations.

In [49]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.loc[
    spearman_by_concentration["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

bucketed_concentration_df = predictions_df.copy()
bucketed_concentration_df["Concentration_bucket"] = pd.Series(
    np.select(
        [
            bucketed_concentration_df["Concentration_nM"] < 0.1,
            bucketed_concentration_df["Concentration_nM"] > 100,
        ],
        ["<0.1 nM", ">100 nM"],
        default="0.1-100 nM",
    ),
    index=bucketed_concentration_df.index,
)

spearman_by_concentration_bucket = grouped_spearman(
    bucketed_concentration_df,
    ["Concentration_bucket"],
    min_samples=20,
)
spearman_by_concentration_bucket.sort_values(["spearman", "mae"], ascending=[True, False])

high_concentration_df = bucketed_concentration_df.loc[
    bucketed_concentration_df["Concentration_bucket"] == ">100 nM"
].copy()

spearman_high_concentration_gene = grouped_spearman(
    high_concentration_df,
    ["group"],
    min_samples=20,
)
spearman_high_concentration_patent = grouped_spearman(
    high_concentration_df,
    ["patent_group"],
    min_samples=20,
)

print(">100 nM by gene")
display(spearman_high_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20))

print(">100 nM by patent")
display(spearman_high_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20))

>100 nM by gene


,group,n_samples,spearman,mae,mean_true,mean_pred
0,MAPT,79,-0.151756,32.326636,59.634177,72.939171
1,HSD17B13,88,-0.050457,30.261644,37.002500,63.678688
2,LAMIN A,44,0.575721,21.404976,87.227273,68.445068


>100 nM by patent


,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,WO2023175091A2,79,-0.151756,32.326636,59.634177,72.939171
1,WO2023091644A2,88,-0.050457,30.261644,37.002500,63.678688
2,HISTORIC_OR_UNKNOWN,44,0.575721,21.404976,87.227273,68.445068


## Concentration + Gene Hotspots

In [50]:
spearman_by_concentration_gene.loc[
    spearman_by_concentration_gene["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,20.00000,APP,170,-0.538620,39.857762,41.005882,61.545261
5,100.00000,INHBE,180,-0.291310,38.886995,62.262167,29.998730
18,100.00000,PCSK9,295,-0.109647,28.799561,54.484407,51.973896
25,0.10000,INHBE,899,-0.028087,22.578421,4.532692,8.107514
26,10.00000,PLN,135,-0.021430,30.574649,50.096000,70.296783
28,10.00000,AGT,1477,-0.017217,37.674691,72.446032,42.085175
29,30.00000,PCSK9,637,-0.015904,30.322148,48.905808,50.007900
30,1.00000,INHBE,1215,-0.005175,26.796183,19.193424,22.153156
35,1.00000,HSD17B13,517,0.041726,19.109784,67.860561,58.181530
39,0.00064,AGT,101,0.054635,9.736001,-3.156733,-0.882927


In [ ]:
spearman_by_concentration_gene.loc[
    spearman_by_concentration_gene["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,20.00000,APP,170,-0.538620,39.857762,41.005882,61.545261
5,100.00000,INHBE,180,-0.291310,38.886995,62.262167,29.998730
18,100.00000,PCSK9,295,-0.109647,28.799561,54.484407,51.973896
25,0.10000,INHBE,899,-0.028087,22.578421,4.532692,8.107514
26,10.00000,PLN,135,-0.021430,30.574649,50.096000,70.296783
28,10.00000,AGT,1477,-0.017217,37.674691,72.446032,42.085175
29,30.00000,PCSK9,637,-0.015904,30.322148,48.905808,50.007900
30,1.00000,INHBE,1215,-0.005175,26.796183,19.193424,22.153156
35,1.00000,HSD17B13,517,0.041726,19.109784,67.860561,58.181530
39,0.00064,AGT,101,0.054635,9.736001,-3.156733,-0.882927


## Concentration + Patent Hotspots

In [51]:
spearman_by_concentration_patent.loc[
    spearman_by_concentration_patent["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
1,20.0,US20220380773A1,170,-0.538620,39.857762,41.005882,61.545261
2,10.0,CN112313335A,182,-0.502275,57.558380,92.877308,35.318928
4,0.1,CN112313335A,184,-0.460491,88.716766,98.657065,9.940299
9,10.0,WO2023091644A2,117,-0.302388,34.769651,58.020085,59.153942
10,100.0,WO2023003922A1,180,-0.291310,38.886995,62.262167,29.998730
15,0.1,CN116515835A,165,-0.221196,25.492194,63.266364,59.218781
32,10.0,US20220117999A1,281,-0.073168,46.581338,28.495302,74.344917
38,0.1,WO2023003922A1,899,-0.028087,22.578421,4.532692,8.107514
41,10.0,WO2023064530A1,135,-0.021430,30.574649,50.096000,70.296783
44,30.0,CN101484588B,637,-0.015904,30.322148,48.905808,50.007900


## Experimental Feature Issues

These summaries highlight which experimental contexts are associated with poor ranking or high error in this XGBoost setup.

In [52]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.loc[
    issue_by_cell_type["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Primary mouse hepatocytes,316,63.330396,74.213671,-0.266484,25.005696,39.466957
1,Non-human hepatocytes,190,25.529923,32.668239,0.009029,22.762105,46.668053
2,Hepa1-6,139,30.090139,36.083563,0.014949,51.316259,70.540382
3,Primary human hepatocytes,2944,27.243324,33.665602,0.090498,50.880149,52.835625
4,Human iPSC-derived cortical neurons,205,32.127767,38.895725,0.159453,42.600976,65.552010
5,Primary Cynomolgus Monkey Hepatocytes,4268,29.174045,36.828112,0.218596,34.462720,36.625072
6,Hela,1906,29.234994,35.444983,0.298587,41.380367,34.533718
7,Hep3B,11967,29.791641,36.866872,0.315200,42.360623,35.992748
9,Huh7,1569,23.913990,29.677834,0.404715,40.877189,38.676968
10,HepG2,1634,22.055373,29.152665,0.414640,40.625918,50.759121


## Concentration Issue Summary

In [53]:
issue_by_concentration.loc[
    issue_by_concentration["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,0.20000,120,47.310777,53.252246,-0.412689,18.021167,9.835164
6,30.00000,769,32.735592,39.428001,-0.106319,45.372458,53.900074
8,2.00000,155,33.801824,39.013665,-0.064763,54.761097,50.667126
13,100.00000,1223,30.929412,38.439841,-0.030193,51.361987,54.994884
14,0.01000,199,28.291036,36.256261,-0.021478,36.840503,15.634002
15,0.01600,114,20.848666,24.255324,-0.013790,21.496842,16.604809
16,0.00064,114,10.525603,13.789906,0.001528,-1.481053,-1.284745
17,20.00000,387,30.264128,37.209406,0.021022,44.191912,56.425499
18,0.05000,296,19.257761,23.206755,0.026346,16.847872,21.794186
19,0.08000,114,24.395394,28.368549,0.074176,45.795614,39.252964


## Time Issue Summary

In [54]:
issue_by_time.loc[
    issue_by_time["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,168.0,205,32.127767,38.895725,0.159453,42.600976,65.552010
1,40.0,197,19.065462,23.558906,0.232958,27.381218,16.958704
2,24.0,24405,29.317897,36.495529,0.302714,41.832551,37.508427
4,72.0,1404,26.285725,31.616429,0.409841,16.028462,38.981285
5,48.0,9157,19.452414,25.270731,0.547636,50.840136,51.071178


## Patent Issue Summary

In [55]:
issue_by_patent.loc[
    issue_by_patent["n_samples"] >= 100
].sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,CN112313335A,366,73.222705,75.538043,-0.809342,95.782978,22.560272
1,US20220380773A1,170,39.857762,46.149778,-0.538620,41.005882,61.545261
4,CN117210468A,225,21.581029,26.368726,-0.027255,25.280444,21.043114
5,WO2023064530A1,135,30.574649,36.520166,-0.021430,50.096000,70.296783
6,WO2023091644A2,1445,26.344493,33.277250,0.005219,48.121391,60.233562
7,TW202321444A,190,25.529923,32.668239,0.009029,22.762105,46.668053
8,CN101484588B,718,28.975280,35.943664,0.024178,49.736769,51.232025
9,CN116801886A,272,35.375873,41.470520,0.034969,50.522426,38.116009
10,WO2022121959A1,100,38.676509,46.615626,0.040914,64.982500,48.505009
12,CN117448322A,132,48.666383,54.449420,0.083548,45.159848,9.537886


## Authorization Issue Summary

In [56]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,225,21.581029,26.368726,-0.027255,25.280444,21.043114
1,Granted,1747,29.214604,35.218797,0.204741,49.487023,45.635624
2,Unknown Status,12741,26.906523,34.379585,0.334404,33.544606,43.977631
3,Withdrawn,1094,25.249910,30.911459,0.386782,30.082541,28.488501
4,Substantive Examination,16154,28.957964,35.818900,0.405584,46.304615,34.662746
5,UNKNOWN,3483,13.820146,17.484416,0.612068,64.647422,63.535412


## Prediction Preview

In [57]:
predictions_df[["group", "Cell_Type", "Concentration_nM", "Time_of_administration_h", "patent_group", "y_true", "y_pred", "residual", "abs_error"]].head(20)

,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,LPA,Hep3B,1.0,24.0,CN108368506A,-1.2,34.769035,-35.969035,35.969035
1,LPA,Hep3B,1.0,24.0,CN108368506A,22.4,0.201030,22.198970,22.198970
2,LPA,Hep3B,1.0,24.0,CN108368506A,29.2,47.724365,-18.524365,18.524365
3,LPA,Hep3B,1.0,24.0,CN108368506A,55.9,45.757290,10.142710,10.142710
4,LPA,Hep3B,1.0,24.0,CN108368506A,75.8,43.617760,32.182240,32.182240
5,LPA,Hep3B,1.0,24.0,CN108368506A,83.4,71.193184,12.206816,12.206816
6,LPA,Hep3B,1.0,24.0,CN108368506A,29.8,22.649752,7.150248,7.150248
7,LPA,Hep3B,1.0,24.0,CN108368506A,72.8,42.235550,30.564450,30.564450
8,LPA,Hep3B,1.0,24.0,CN108368506A,71.0,51.123627,19.876373,19.876373
9,LPA,Hep3B,1.0,24.0,CN108368506A,17.5,47.982811,-30.482811,30.482811
